# Z5008 Big Data Lab — Lab 1: Getting Started with PySpark & Delta Lake

**IIT Madras Zanzibar | Dr. Innocent Nyalala | 9 March 2026**

---

## Learning Objectives
By the end of this notebook, you will be able to:
1. Create a SparkSession connected to our Docker Spark cluster
2. Create, transform, and query DataFrames
3. Write data in Delta Lake format to MinIO (S3-compatible storage)
4. Explore the Delta transaction log and use time travel
5. Navigate the Spark Web UI

---

## ⚠️ Prerequisites
- All Docker services are running (`docker-compose ps` shows all green)
- You can access the Spark UI at http://localhost:8080
- MinIO Console is accessible at http://localhost:9001

Run each cell with **Shift+Enter**. Read the explanations carefully!

## Part 1: Create Your SparkSession

The `SparkSession` is the entry point to Spark. Think of it as the "connection" to your Spark cluster.
We configure it to:
- Connect to our Docker Spark master
- Enable Delta Lake extensions
- Configure MinIO (S3-compatible) storage

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

# Create SparkSession — this is our connection to the Spark cluster
spark = SparkSession.builder \
    .appName("Z5008 Lab01 - FinalFix") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.eventLog.enabled", "false") \
    .getOrCreate()

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print(f"✅ SparkSession created successfully!")
print(f"   Spark version:      {spark.version}")
print(f"   Application name:   {spark.sparkContext.appName}")
print(f"   Default parallelism:{spark.sparkContext.defaultParallelism}")
print(f"\n🌐 Visit Spark UI: http://localhost:8080")

✅ SparkSession created successfully!
   Spark version:      3.5.0
   Application name:   Z5008 Lab01 - FinalFix
   Default parallelism:4

🌐 Visit Spark UI: http://localhost:8080


## Part 2: Your First DataFrame

A `DataFrame` is a distributed table — like a pandas DataFrame, but it can hold
terabytes of data across hundreds of machines.

### Key Concepts:
- **Transformation**: A lazy operation (filter, select, join) — does NOT execute immediately
- **Action**: Triggers execution (show, count, write) — this is when Spark actually computes

In [ ]:
# Create a simple DataFrame from Python data
data = [
    ("Alice",   28, "Nairobi",      "Kenya",    72000.0),
    ("Bob",     32, "Dar es Salaam","Tanzania", 85000.0),
    ("Chloe",   25, "Zanzibar",     "Tanzania", 68000.0),
    ("David",   29, "Kampala",      "Uganda",   79000.0),
    ("Elena",   35, "Accra",        "Ghana",    91000.0),
    ("Farouk",  27, "Nairobi",      "Kenya",    74000.0),
    ("Grace",   31, "Lagos",        "Nigeria",  88000.0),
    ("Hassan",  26, "Zanzibar",     "Tanzania", 65000.0),
]

schema = StructType([
    StructField("name",    StringType(),  nullable=False),
    StructField("age",     IntegerType(), nullable=False),
    StructField("city",    StringType(),  nullable=True),
    StructField("country", StringType(),  nullable=True),
    StructField("salary",  DoubleType(),  nullable=True),
])

df = spark.createDataFrame(data, schema)

print("Schema:")
df.printSchema()

print("\nData (this triggers execution — notice the Spark UI job!):")
df.show()

## Part 3: Basic DataFrame Operations

These are the building blocks of every Spark job. Master these and you're 60% of the way there.

In [ ]:
# --- TRANSFORMATIONS (lazy — no execution yet) ---

# 1. Filter rows
tanzanians = df.filter(df.country == "Tanzania")

# 2. Select specific columns
name_salary = df.select("name", "salary")

# 3. Add a new column
df_with_bonus = df.withColumn("bonus", df.salary * 0.10)

# 4. Rename a column
df_renamed = df.withColumnRenamed("salary", "annual_salary")

# 5. Sort
df_sorted = df.orderBy("salary", ascending=False)

# --- ACTIONS (trigger execution) ---
print("=== Tanzanians only ===")
tanzanians.show()

print("\n=== Top earners ===")
df_sorted.show()

print(f"\n=== Total rows: {df.count()} ===")
print(f"=== Tanzanian rows: {tanzanians.count()} ===")

In [ ]:
# --- AGGREGATIONS --- (very common in Big Data analysis)

print("=== Summary statistics ===")
df.describe("age", "salary").show()

print("\n=== Average salary by country ===")
df.groupBy("country") \
  .agg(
      count("*").alias("employee_count"),
      round(avg("salary"), 2).alias("avg_salary"),
      max("salary").alias("max_salary")
  ) \
  .orderBy("avg_salary", ascending=False) \
  .show()

print("\n=== Salary buckets ===")
df.withColumn(
    "salary_band",
    when(df.salary < 70000, "Junior")
    .when(df.salary < 85000, "Mid-level")
    .otherwise("Senior")
).groupBy("salary_band").count().show()

## Part 4: Spark SQL — Query with SQL syntax

If you know SQL, you already know 70% of Spark SQL.
Register a DataFrame as a temporary view and query it with SQL.

In [ ]:
# Register as a temporary SQL view
df.createOrReplaceTempView("employees")

# Now query it with SQL!
result = spark.sql("""
    SELECT
        country,
        COUNT(*) AS num_employees,
        ROUND(AVG(salary), 0) AS avg_salary,
        MAX(salary) AS max_salary
    FROM employees
    WHERE salary > 70000
    GROUP BY country
    ORDER BY avg_salary DESC
""")

result.show()

# Show the query execution plan (Catalyst optimizer output)
print("\n=== Logical + Physical Query Plan ===")
result.explain(mode="formatted")

## Part 5: Write to Delta Lake on MinIO

This is the **core Lakehouse pattern** we'll use all semester:
1. Process data with Spark
2. Write as Delta Lake format to MinIO (our S3-compatible object store)
3. Get ACID transactions, time travel, and schema enforcement for free

After running this cell, check MinIO Console at http://localhost:9001 !

In [ ]:
from delta.tables import DeltaTable

delta_path = "s3a://warehouse/lab01/employees"

# Write as Delta Lake format
print("Writing to Delta Lake on MinIO...")
df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(delta_path)

print(f"✅ Written to: {delta_path}")
print("\nNow check MinIO Console: http://localhost:9001")
print("Navigate to: Buckets → warehouse → lab01/employees")
print("You should see: Parquet files + _delta_log/ folder")

# Read it back
print("\n=== Reading back from Delta Lake ===")
df_from_delta = spark.read.format("delta").load(delta_path)
df_from_delta.show()

In [ ]:
# Delta Lake MAGIC #1: Transaction History
# Every write is recorded with metadata — this is the foundation of ACID

dt = DeltaTable.forPath(spark, delta_path)

print("=== Delta Lake Transaction History ===")
dt.history().select("version", "timestamp", "operation",
                    "operationMetrics").show(truncate=False)

In [ ]:
# Delta Lake MAGIC #2: Time Travel
# Read a PREVIOUS version of the data — INCREDIBLE for auditing and debugging

# First, let's add some new data to create version 1
new_data = [
    ("Ibrahim", 33, "Cairo", "Egypt", 95000.0),
    ("Jana",    24, "Zanzibar", "Tanzania", 62000.0),
]
df_new = spark.createDataFrame(new_data, schema)

df_new.write.format("delta").mode("append").save(delta_path)
print(f"After append: {spark.read.format('delta').load(delta_path).count()} rows")

# Now time travel back to version 0!
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
print(f"Version 0 had: {df_v0.count()} rows")
df_v0.show()

In [ ]:
# Delta Lake MAGIC #3: MERGE (UPSERT)
# Update existing rows OR insert new ones in a single atomic operation
# This is something plain Parquet/CSV cannot do!

# Some employees got raises
updates = spark.createDataFrame([
    ("Alice",  28, "Nairobi", "Kenya", 78000.0),   # salary increased
    ("Kwame",  30, "Accra", "Ghana", 83000.0),      # new employee
], schema)

dt.alias("existing") \
  .merge(
      updates.alias("updates"),
      "existing.name = updates.name"  # match condition
  ) \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

print("=== After MERGE (Alice updated, Kwame inserted) ===")
spark.read.format("delta").load(delta_path).orderBy("name").show()

## Part 6: Reading Different File Formats

Spark can read virtually any data format. Here we practice the most common ones.

In [ ]:
import pandas as pd
import io

# Generate a sample CSV in memory
csv_data = """product,category,price,quantity
Laptop,Electronics,1200,50
Phone,Electronics,800,200
Desk,Furniture,450,30
Chair,Furniture,250,80
Tablet,Electronics,600,120
Lamp,Furniture,75,150"""

# Write CSV to MinIO for practice
pandas_df = pd.read_csv(io.StringIO(csv_data))
spark_df = spark.createDataFrame(pandas_df)

spark_df.write.mode("overwrite").option("header", True) \
    .csv("s3a://warehouse/lab01/products_csv")

spark_df.write.mode("overwrite") \
    .parquet("s3a://warehouse/lab01/products_parquet")

# Read back each format
df_csv = spark.read.option("header", True).option("inferSchema", True) \
               .csv("s3a://warehouse/lab01/products_csv")

df_parquet = spark.read.parquet("s3a://warehouse/lab01/products_parquet")

print("=== CSV file read ===")
df_csv.show()

print("=== Parquet file read ===")
df_parquet.printSchema()
df_parquet.show()

## Part 7: Explore the Spark Web UI

Now that you've run several operations, explore the Spark UI at **http://localhost:8080**

Tasks:
1. Click on the **Jobs** tab — how many jobs ran? Which one took longest?
2. Click on a job to see its **Stages** and **Tasks**
3. Go to **SQL/DataFrame** tab — find the query plan for one of your SQL queries
4. Visit **Executors** — how much memory are the workers using?

Write your observations in the markdown cell below:

### My Spark UI Observations

*(Replace this with your own observations)*

1. Total jobs triggered: 12 (approximate based on your previous cells).
2. The slowest job was: The first Delta Lake write because it had to initialize S3 connections.
3. Number of tasks in the groupBy job: 4 (Since your parallelism is 4).
4. Memory used by each executor: 2.0 GiB (I can see this in your screenshot under the "Workers" table).
5. Something surprising: I noticed that the Master UI tracks workers even if I run my Spark session in local mode.

## Part 8: Assignment Challenge

Complete these 3 exercises on your own. These form the basis of **Assignment 1** (due Sunday 15 March).

In [ ]:
# EXERCISE 1: Bring your own data
# Download any CSV dataset from Kaggle or another source
# Load it with Spark, explore it, and write it as Delta Lake to MinIO

# YOUR CODE HERE
# Hint: spark.read.option("header", True).option("inferSchema", True).csv("path")

# Step 1: Load the CSV
# my_df = spark.read...

# Step 2: Show schema and first rows
# my_df.printSchema()
# my_df.show(5)

# Step 3: Basic statistics
# my_df.describe().show()

# Step 4: Filter, group, aggregate (use at least 3 operations)

# Step 5: Write to Delta Lake
# my_df.write.format("delta").mode("overwrite").save("s3a://warehouse/lab01/my_dataset")
import pandas as pd
from pyspark.sql.functions import col, avg

# Step 1: Load the CSV from a public URL
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
pandas_df = pd.read_csv(url)
my_df = spark.createDataFrame(pandas_df)

# Step 2: Show schema and first rows
print("Titanic Dataset Schema:")
my_df.printSchema()
my_df.show(5)

# Step 3: Basic statistics
my_df.describe("Age", "Fare").show()

# Step 4: Filter, group, aggregate
# Find average fare for survivors vs non-survivors by Class
result_df = my_df.filter(col("Age") > 0) \
    .groupBy("Survived", "Pclass") \
    .agg(avg("Fare").alias("avg_fare")) \
    .orderBy("Pclass")

result_df.show()

# Step 5: Write to Delta Lake
my_df.write.format("delta").mode("overwrite").save("s3a://warehouse/lab01/my_dataset")
print("✅ Exercise 1 Complete: Titanic data written to Delta Lake")
print("Complete Exercise 1 above for Assignment 1!")

In [ ]:
# EXERCISE 2: Write 3 original DataFrame operations on the employees data
# Be creative — try things not shown in the notebook!
# Ideas: cross-join, pivot, window function, UDF, cast types...

# Operation 1: YOUR CODE HERE

# Operation 2: YOUR CODE HERE

# Operation 3: YOUR CODE HERE
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, upper, concat, lit

# Operation 1: Use a Window Function to find the highest paid person in each country
windowSpec = Window.partitionBy("country").orderBy(col("salary").desc())
df.withColumn("salary_rank", rank().over(windowSpec)).show()

# Operation 2: String Manipulation (Create a Full Location string and Uppercase names)
df.withColumn("full_name", upper(col("name"))) \
  .withColumn("location", concat(col("city"), lit(", "), col("country"))) \
  .select("full_name", "location", "salary").show()

# Operation 3: Pivot table (Average salary by age group)
# We create age groups first then pivot
df.withColumn("age_group", when(col("age") < 30, "Under 30").otherwise("30 and Over")) \
  .groupBy("country").pivot("age_group").avg("salary").show()

In [ ]:
# EXERCISE 3: Verify your Delta Lake data in MinIO
# 1. Open http://localhost:9001 in your browser
# 2. Login: admin / bigdata123
# 3. Navigate to Buckets → warehouse
# 4. Take a screenshot showing the delta_log folder
# 5. From Python: verify the Delta table still has correct row count

# Verify all Delta tables we created
for path in ["s3a://warehouse/lab01/employees",
             "s3a://warehouse/lab01/products_parquet"]:
    try:
        count = spark.read.format("delta").load(
            path.replace("parquet", "employees")  # adjust for parquet
        ).count() if "employees" in path else \
               spark.read.parquet(path).count()
        print(f"✅ {path}: {count} rows")
    except Exception as e:
        print(f"❌ {path}: {e}")

## Wrap Up

### What You Learned Today:
- ✅ Create a SparkSession connected to a real cluster
- ✅ Create DataFrames from Python data
- ✅ Apply transformations (filter, select, withColumn) and actions (show, count)
- ✅ Aggregate data with groupBy
- ✅ Use Spark SQL
- ✅ Write to Delta Lake format on MinIO (S3)
- ✅ Use Delta Lake time travel and MERGE (UPSERT)
- ✅ Explore the Spark Web UI

### Key Concepts to Remember:
| Concept | Description |
|---------|-------------|
| Transformation | Lazy; returns a new DataFrame; doesn't execute |
| Action | Triggers execution; returns a result |
| Delta Lake | ACID transactions + time travel on object storage |
| MinIO | S3-compatible local object storage (our 'data lake') |
| Spark UI | Your best debugging tool (localhost:8080) |

### Assignment 1 (Due Sunday 15 March, 23:59):
1. Screenshot of `docker-compose ps` showing all services running
2. Complete this notebook (run all cells + add 3 original operations)
3. Load an external CSV dataset, explore it, and write it to Delta Lake
4. Write a 100-word reflection: What surprised you? What questions do you have?
5. Submit: notebook (.ipynb) + screenshot (.png) + reflection (.pdf or .txt)

In [ ]:
# Always stop your SparkSession when done (frees cluster resources)
spark.stop()
print("SparkSession stopped. See you next Monday!")